# 4-2 · 한국어 도메인 감성어 선정 시스템 (4단계)

> **목표:** 3단계의 후보와 실제 리뷰 예문을 브라우저에서 살펴보고, **AI 점수 추천 → 사람의 검토·수정 → CSV 내보내기 → KNU 확장판 생성** 흐름을 만든다.

## 작업의 경계

- **원본 데이터:** `4-2/data/`만 사용한다. 노트북은 `4-2/notebooks/`에 놓는다.
- **입력:** 3단계 `domain_lexicon_curation_*.csv`와 `label_context_examples_*.json`, KNU `SentiWord_info.json`.
- **중간 산출물:** `ai_review_request_*.csv` → **ChatGPT 대화에 첨부**하여 추천 점수를 받는다.
- **검토:** 노트북에서 생성한 **독립 실행형 HTML**을 브라우저로 열어 추천 결과를 불러오고, 단어별로 승인·수정·제외한다.
- **내보내기:** 브라우저 버튼으로 `selected_domain_sentiment.csv`를 다운로드한다.
- **마무리:** 다운로드한 CSV를 지정한 `data/` 폴더로 옮긴 뒤 마지막 셀에서 **KNU 확장판 JSON**을 새 버전으로 생성한다.

## 중요한 개념

- 긍정 리뷰에 자주 등장했다는 사실만으로 그 단어에 긍정 점수를 주면 안 된다.
- `영화`, `연기`, `스토리` 등은 주제어(Aspect)일 수 있다.
- `너무`, `정말`은 일반 감성어보다 강조어(Modifier)로 처리할 가능성이 높다.
- `않다`, `안`은 부정어(Negator) 규칙의 대상이다.
- **AI 추천은 후보 CSV를 실제로 ChatGPT에 제공한 뒤에만 받을 수 있다.** 이번 노트북은 데이터 없이 임의의 추천 점수를 만들지 않는다.
- **`unmatched`는 이전 단계의 매칭 결과**일 뿐, KNU에 완전히 없는 표현임을 보증하지 않는다.

## 0. 경로와 의존성 확인

- `pandas`와 `IPython`만 사용한다. 3단계에서 형태소 분석한 결과를 재사용하므로 **여기서는 Java/Okt를 다시 실행하지 않는다.**
- CSV·JSON은 모두 `data/`에서 읽고, HTML 및 중간 CSV도 `data/outputs/sentiment_vocab/` 아래에 만든다.
- VS Code 노트북의 현재 작업 디렉터리가 루트 또는 `notebooks/`여도 작동하게 루트를 탐색한다.

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
import csv
import hashlib
import json
import math
import os
import re
import sys
import webbrowser

import pandas as pd

ROOT = next(
    (p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("4-2/ 또는 4-2/notebooks/에서 실행해야 합니다.")
DATA = ROOT / "data"
OUT = DATA / "outputs" / "sentiment_vocab"
OUT.mkdir(parents=True, exist_ok=True)
KNU_JSON = DATA / "lexicons" / "ko" / "SentiWord_info.json"
if not KNU_JSON.is_file():
    raise FileNotFoundError(f"KNU 원본 파일이 없습니다: {KNU_JSON}")

print("프로젝트:", ROOT)
print("Python:", sys.executable)
print("출력 디렉터리:", OUT.relative_to(ROOT))
print("pandas:", pd.__version__)

## 1. 3단계 검토 CSV 로드

- 3단계의 `domain_lexicon_curation_*.csv` 중 하나를 선택한다.
- 둘 이상 있으면 **가장 최근 파일을 자동 선택하되 선택된 이름을 반드시 출력**한다. 다른 범위를 사용하려면 `CURATION_NAME`을 직접 지정한다.
- `(token, pos)` 조합은 고유 키다. 동일 키가 중복되면 그대로 진행하지 않는다.
- `knu_score`가 비어 있다는 것과 0점이라는 것은 다르다.

In [ ]:
# 원하는 파일명이 있으면 문자열로 지정. None이면 가장 최근 생성/수정된 후보 파일 사용.
CURATION_NAME = None

available = sorted(OUT.glob("domain_lexicon_curation_*.csv"), key=lambda p: (p.stat().st_mtime, p.name), reverse=True)
if not available:
    raise FileNotFoundError("3단계에서 먼저 domain_lexicon_curation_*.csv를 생성하세요.")
print("발견된 후보 CSV:")
for p in available:
    print(" -", p.name)
CURATION_PATH = (OUT / CURATION_NAME) if CURATION_NAME else available[0]
if not CURATION_PATH.is_file():
    raise FileNotFoundError(CURATION_PATH)
print("\n이번에 사용할 파일:", CURATION_PATH.name)

curation = pd.read_csv(CURATION_PATH, encoding="utf-8-sig", dtype={"token": "string", "pos": "string"})
required_cols = {"token", "pos", "review_count", "token_count", "match_status"}
missing = required_cols - set(curation.columns)
if missing:
    raise ValueError(f"후보 CSV에 필요한 컬럼이 없습니다: {sorted(missing)}")
for col in ["token", "pos", "match_status"]:
    curation[col] = curation[col].astype("string").fillna("")
if curation[["token", "pos"]].duplicated().any():
    raise ValueError("(token,pos)가 중복되어 후보를 구분할 수 없습니다.")
if curation["token"].str.strip().eq("").any():
    raise ValueError("빈 token이 있습니다.")
for col in ["review_count", "token_count", "positive_review_hits", "negative_review_hits", "log2_class_ratio", "knu_score"]:
    if col not in curation:
        curation[col] = pd.NA
    curation[col] = pd.to_numeric(curation[col], errors="coerce")
for col in ["decision", "role", "reason"]:
    if col not in curation:
        curation[col] = ""
    curation[col] = curation[col].astype("string").fillna("")
if "custom_score" not in curation:
    curation["custom_score"] = pd.NA
curation["custom_score"] = pd.to_numeric(curation["custom_score"], errors="coerce")
print(f"후보 {(len(curation)):,}행")
display(curation[["token", "pos", "review_count", "match_status", "log2_class_ratio"]].head(10))

## 2. KNU 원본과 긍정·부정 예문 합치기

- KNU JSON은 **읽기 전용**으로 사용한다.
- 이전 단계에서 만든 `label_context_examples_..._v1.json`을 자동으로 찾는다.
- 예문이 없는 단어도 정상이다. 3단계에서 문맥 수집 대상이 아니었을 수 있다.
- KNU에 같은 **문자열**이 있는 후보는 확장 사전에 넣을 때 충돌할 수 있으므로 별도 표시한다.

In [ ]:
knu_records = json.loads(KNU_JSON.read_text(encoding="utf-8-sig"))
if not isinstance(knu_records, list) or not all(isinstance(item, dict) and "word" in item and "polarity" in item for item in knu_records):
    raise ValueError("KNU JSON의 구조가 예상과 다릅니다.")
knu_words = {str(item["word"]) for item in knu_records}

# 예: domain_lexicon_curation_full_balanced_20000_seed42.csv
#       -> label_context_examples_balanced_20000_seed42_v1.json
# SOURCE_SCOPE은 보통 full이며 3단계에서 사용한 값을 유지한다.
SOURCE_SCOPE = "full"
prefix = f"domain_lexicon_curation_{SOURCE_SCOPE}_"
if CURATION_PATH.name.startswith(prefix):
    scope_label = CURATION_PATH.stem[len(prefix):]
    example_file = OUT / f"label_context_examples_{scope_label}_v1.json"
else:
    example_file = None
if example_file is None or not example_file.is_file():
    print("⚠ 대응하는 예문 JSON을 찾지 못했습니다. 예문 없이도 검토할 수 있습니다.")
    context_examples = {}
else:
    context_examples = json.loads(example_file.read_text(encoding="utf-8"))
    print("예문 파일:", example_file.name)

# 문맥은 전체 리뷰의 레이블이며 특정 단어 하나의 감성 정답은 아니다.
def examples_for(token, pos, label, limit=3):
    bucket = context_examples.get(f"{token}\t{pos}", {})
    return [str(x.get("text", "")) for x in bucket.get(str(label), [])[:limit]]

curation["knu_exact_word_exists"] = curation["token"].map(lambda t: str(t) in knu_words)
print("KNU 원본 항목:", len(knu_records))
print("후보와 KNU 원본의 동일 문자열 중복:", int(curation["knu_exact_word_exists"].sum()))

## 3. ChatGPT에 제공할 중간 CSV 생성

추천 점수는 이 **중간 CSV를 ChatGPT에 첨부한 다음** 받아야 해. 노트북 자체에서는 ChatGPT를 호출하지 않는다.

- **보낼 내용:** `(token, pos)`, 빈도, 긍정·부정 리뷰 출현 수, 편중 지표, 실제 예문.
- **ChatGPT가 작성할 열:** `suggested_score` (`-2`, `-1`, `0`, `1`, `2` 또는 공란), `suggested_role`, `suggested_reason`.
- **추천 원칙:** 근거가 약하거나 중립적인 단어는 **0점 또는 공란**으로 두고, `sentiment`가 아닌 `aspect`/`modifier`/`negator` 역할을 표시한다.
- **주의:** `log2_class_ratio`는 감성 점수가 아니다. 원문 예시를 확인해야 한다.
- **표본:** 리뷰 빈도 상위 + 긍정/부정 편중이 큰 표현을 섞어 만든다. `MAX_AI_ROWS`를 조절할 수 있다.

### 추천 CSV 작성 요청 예시

> 첨부한 CSV의 `suggested_score`, `suggested_role`, `suggested_reason`를 채워줘. 영화 리뷰 도메인에 **고유한 감성 표현**을 우선하고, 단순 주제어·부정어·강조어는 감성 점수로 채우지 말아줘. 문맥이 불분명하면 보류하고, 입력 CSV의 `token`, `pos` 및 다른 데이터는 바꾸지 말아줘. 작성된 CSV를 파일로 돌려줘.

In [ ]:
# AI에 보내는 행 수. 크게 만들면 대화 파일 크기와 검토량이 늘어난다.
MAX_AI_ROWS = 300

# 주제어만 몰리지 않도록 빈도 상위 1/3, 긍정·부정 편중 상위 각 1/3을 합친다.
rankable = curation.loc[~curation["knu_exact_word_exists"]].copy()
rankable = rankable.loc[rankable["match_status"].eq("unmatched")].copy()
rankable["review_count"] = rankable["review_count"].fillna(0)
rankable["log2_class_ratio"] = rankable["log2_class_ratio"].fillna(0)
per_group = max(1, math.ceil(MAX_AI_ROWS / 3))
most_frequent = rankable.nlargest(per_group, "review_count")
MIN_AI_SAMPLE_HITS = 15  # 소수 리뷰에서만 등장한 극단적 비율 후보를 제외
strong_enough = rankable.loc[
    (rankable["positive_review_hits"].fillna(0) + rankable["negative_review_hits"].fillna(0)) >= MIN_AI_SAMPLE_HITS
]
positive_lean = strong_enough.loc[strong_enough["log2_class_ratio"] > 0].nlargest(per_group, "log2_class_ratio")
negative_lean = strong_enough.loc[strong_enough["log2_class_ratio"] < 0].nsmallest(per_group, "log2_class_ratio")
ai_pool = pd.concat([most_frequent, positive_lean, negative_lean]).drop_duplicates(["token", "pos"])
ai_pool = ai_pool.head(MAX_AI_ROWS).copy()

ai_pool["positive_examples"] = [" | ".join(examples_for(t, p, 1, limit=2)) for t, p in zip(ai_pool.token, ai_pool.pos)]
ai_pool["negative_examples"] = [" | ".join(examples_for(t, p, 0, limit=2)) for t, p in zip(ai_pool.token, ai_pool.pos)]
for col in ["suggested_score", "suggested_role", "suggested_reason"]:
    ai_pool[col] = ""  # 여기서는 점수를 추측해서 채우지 않는다.

AI_REQUEST_CSV = OUT / f"ai_review_request_{CURATION_PATH.stem.removeprefix('domain_lexicon_curation_')}.csv"
# AI 요청 CSV는 입력/전송용 중간 생성물이므로, 동일 노트북 설정으로 재생성 가능하다.
request_cols = ["token", "pos", "token_count", "review_count", "positive_review_hits",
                "negative_review_hits", "log2_class_ratio", "positive_examples", "negative_examples",
                "suggested_score", "suggested_role", "suggested_reason"]
ai_pool[request_cols].to_csv(AI_REQUEST_CSV, index=False, encoding="utf-8-sig")
print("ChatGPT에 첨부할 파일:", AI_REQUEST_CSV.relative_to(ROOT))
print("후보 수:", len(ai_pool))
display(ai_pool[request_cols].head(8))
display(FileLink(str(AI_REQUEST_CSV)))

## 4. 브라우저 검토 화면 만들기

**이 셀부터는 HTML + JavaScript만 사용해 로컬 브라우저에서 편집한다.**

- 표에서 후보 검색, 품사 필터, 빈도·긍정/부정 편중 순 정렬.
- 후보 클릭 → 긍정·부정 리뷰 예문과 통계 확인 → 추천 점수 확인 → **수동 수정 또는 승인**.
- **AI 추천 불러오기:** ChatGPT가 돌려준 CSV를 HTML의 `AI 추천 CSV 불러오기`로 선택한다. 키는 `(token, pos)`.
- **검토 저장:** `검토 전체 CSV 다운로드`로 지금까지의 모든 작업을 보존한다.
- **최종 선정:** `선정 30개 이상 CSV 다운로드`를 눌러 **승인된 감성어만** 내보낸다.
- HTML은 서버가 없는 **독립 파일**이며 브라우저에서 변경한 내용이 파이썬 데이터 파일에 실시간 저장되지는 않는다. CSV 다운로드가 저장 단계다.
- 브라우저 새로고침 시 편집 내용이 사라질 수 있으므로, 검토 중에도 CSV를 자주 다운로드해두자. 내려받은 검토 CSV는 다시 불러올 수 있다.

### 추천 점수를 아직 받지 않았다면

- 화면에는 추천 점수가 `미입력`으로 표시되는 것이 정상이다.
- ChatGPT가 작성한 CSV를 받은 후 점수를 가져오거나, 직접 점수를 입력할 수 있다.

In [ ]:
# 이 HTML은 CSS/JavaScript까지 포함한 독립 실행형 검토 UI 템플릿이다.
# 아래 문자열의 __CANDIDATE_JSON__ 부분에 Python에서 후보 데이터를 주입한다.
# 데이터는 로컬 HTML에 포함되므로 외부 서버나 인터넷이 필요하지 않다.
HTML_TEMPLATE = r'''
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>미션 4-2 | 도메인 감성어 검토</title>
<style>
:root{font-family:-apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo","Noto Sans KR",Segoe UI,sans-serif;color:#182334;background:#f4f6f9;line-height:1.55}
*{box-sizing:border-box}body{margin:0}.wrap{max-width:1440px;margin:auto;padding:24px}
header{background:#17253a;color:#fff;padding:22px 0}header h1{font-size:25px;margin:0 0 8px}header p{color:#d4deeb;margin:0}
.badge{font-size:12px;background:#e7edf5;border-radius:6px;padding:4px 7px;color:#4c5970}
.stats{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:18px 0}
.stat,.panel{background:white;border:1px solid #d9e0e9;border-radius:11px;box-shadow:0 2px 10px #263b5010}.stat{padding:16px}.stat strong{font-size:23px;display:block}.stat span{font-size:13px;color:#607086}
.toolbar{display:flex;flex-wrap:wrap;align-items:center;gap:10px;margin-bottom:12px}.toolbar button,.toolbar label.file{cursor:pointer}
button,input,select,textarea{font:inherit}button{border:1px solid #bac7d5;background:#fff;border-radius:7px;padding:8px 11px;color:#26384e;cursor:pointer}
button.primary{background:#2256a0;border-color:#2256a0;color:#fff}button:disabled{opacity:.5;cursor:not-allowed}button:hover:not(:disabled){filter:brightness(.96)}
input[type=file]{display:none}.file{display:inline-block;border:1px solid #9faec2;border-radius:7px;padding:8px 11px;background:#ecf2f9;font-size:14px}
.workspace{display:grid;grid-template-columns:minmax(320px,45%) minmax(370px,55%);gap:14px;align-items:start}
.panel{padding:17px;min-width:0}.panel h2{font-size:17px;margin:0 0 12px}.filters{display:flex;gap:8px;flex-wrap:wrap;margin-bottom:11px}
.filters>*{min-width:0;padding:8px;border-radius:7px;border:1px solid #cbd5e2;background:#fff}.filters input{flex:1;min-width:140px}
.tablewrap{overflow:auto;max-height:660px}table{border-collapse:collapse;width:100%;font-size:13px}th{background:#eff3f8;color:#40536d;text-align:left;position:sticky;top:0;z-index:1}th,td{padding:9px 8px;border-bottom:1px solid #e7ebf0;white-space:nowrap}tr[data-key]{cursor:pointer}tr[data-key]:hover{background:#eef5ff}tr.current{background:#e4eeff}td.num{text-align:right;font-variant-numeric:tabular-nums}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:12px}.field{margin-bottom:12px}.field label{font-size:12px;color:#53657b;display:block;margin-bottom:5px}.field input,.field select,.field textarea{width:100%;border:1px solid #c4cedb;padding:9px;border-radius:7px;background:white}.field textarea{min-height:90px;resize:vertical}
.muted{color:#6c7c90;font-size:13px}.warn{color:#aa5e10}.good{color:#126d53}.neg{color:#ae3940}.examples{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin:10px 0}.examples>div{background:#f7f9fc;border:1px solid #e0e8f2;border-radius:8px;padding:10px;min-height:110px}.examples h3{font-size:13px;margin:0 0 8px}.examples p{font-size:13px;margin:6px 0;border-bottom:1px solid #e5eaf2;padding-bottom:6px;white-space:pre-wrap;word-break:break-all}
.pager{display:flex;justify-content:space-between;align-items:center;margin:12px 0;font-size:13px;color:#526278}.hint{padding:12px 14px;background:#e9eff8;border-radius:8px;font-size:13px;margin-bottom:14px;color:#334962}
.notice{min-height:26px;font-size:13px;margin-top:8px}.note{font-size:12px;color:#69788b}.flex{display:flex;gap:8px;flex-wrap:wrap}
hr{border:0;border-top:1px solid #e4eaf2;margin:16px 0}
@media(max-width:960px){.workspace{grid-template-columns:1fr}.stats{grid-template-columns:repeat(2,1fr)}}
@media(max-width:500px){.wrap{padding:13px}.examples,.grid2{grid-template-columns:1fr}}
</style>
</head>
<body>
<header><div class="wrap"><h1>미션 4-2 · 도메인 감성어 검토</h1><p>후보 탐색 → AI 점수 추천 불러오기 → 사람의 승인/수정 → CSV 다운로드</p></div></header>
<main class="wrap">
<div class="hint"><strong>저장 방식:</strong> 이 페이지는 로컬 HTML입니다. 수정한 데이터는 브라우저에서만 유지되므로 <strong>검토 전체 CSV</strong>를 주기적으로 다운로드하세요. 최종 30개를 선택하면 <strong>선정 CSV</strong>를 다운로드하고 노트북 마지막 셀에서 병합합니다.</div>
<div class="stats">
<div class="stat"><span>총 후보</span><strong id="countAll">-</strong></div>
<div class="stat"><span>AI 추천 입력</span><strong id="countSuggested">-</strong></div>
<div class="stat"><span>승인된 감성어</span><strong id="countSelected">-</strong></div>
<div class="stat"><span>30개 목표 잔여</span><strong id="countRemaining">-</strong></div>
</div>
<div class="toolbar">
<label class="file">AI 추천 CSV 불러오기<input id="aiFile" type="file" accept=".csv,text/csv"></label>
<label class="file">이전 검토 CSV 불러오기<input id="resumeFile" type="file" accept=".csv,text/csv"></label>
<button id="saveReview">검토 전체 CSV 다운로드</button>
<button id="saveSelected" class="primary">선정 30개 이상 CSV 다운로드</button>
</div>
<div id="notice" class="notice" role="status" aria-live="polite"></div>
<div class="workspace">
<section class="panel">
<h2>후보 목록 <span class="badge" id="visibleCount"></span></h2>
<div class="filters">
<input id="search" placeholder="단어 검색" aria-label="단어 검색">
<select id="posFilter" aria-label="품사"><option value="">전체 품사</option></select>
<select id="stateFilter" aria-label="검토 상태"><option value="all">모든 상태</option><option value="review">미검토/보류</option><option value="add">추가 승인</option><option value="reject">제외</option><option value="suggested">AI 추천 있음</option></select>
<select id="sort" aria-label="정렬"><option value="freq">출현 리뷰 수 내림차순</option><option value="positive">긍정 편중 순</option><option value="negative">부정 편중 순</option><option value="alpha">가나다순</option></select>
</div>
<div class="tablewrap"><table><thead><tr><th>단어</th><th>품사</th><th>리뷰</th><th>AI</th><th>최종</th><th>상태</th></tr></thead><tbody id="candidateTable"></tbody></table></div>
<div class="pager"><button id="prevPage">이전</button><span id="pageInfo"></span><button id="nextPage">다음</button></div>
<p class="note">후보는 `(형태소, 품사)` 단위다. 점수 없음과 0점은 다르며, KNU의 미등록 판정도 완전하지 않을 수 있다.</p>
</section>
<section class="panel">
<div id="emptyEditor" class="muted">왼쪽 후보를 선택해 검토하세요.</div>
<div id="editor" hidden>
<h2><span id="wordTitle"></span> <span class="badge" id="posTitle"></span></h2>
<p class="muted" id="metrics"></p>
<p class="warn" id="knuWarning"></p>
<div class="examples"><div><h3 class="good">긍정 리뷰 예문</h3><div id="positiveExamples"></div></div><div><h3 class="neg">부정 리뷰 예문</h3><div id="negativeExamples"></div></div></div>
<hr>
<div class="grid2">
<div class="field"><label>AI 추천 점수</label><div id="suggestedScore" class="hint">미입력</div></div>
<div class="field"><label>AI 추천 역할</label><div id="suggestedRole" class="hint">미입력</div></div>
</div>
<div class="field"><label>AI 추천 근거</label><div id="suggestedReason" class="muted">추천 CSV를 불러온 뒤 표시됩니다.</div></div>
<div class="flex"><button id="applySuggestion">AI 추천을 내 선택에 적용</button></div>
<hr>
<div class="grid2">
<div class="field"><label for="decision">검토 상태</label><select id="decision"><option value="review">보류 / 미검토</option><option value="add">추가 승인</option><option value="reject">제외</option></select></div>
<div class="field"><label for="role">표현의 역할</label><select id="role"><option value="">선택 전</option><option value="sentiment">감성어</option><option value="aspect">대상/주제어</option><option value="modifier">강조어/수식어</option><option value="negator">부정어</option><option value="other">기타</option></select></div>
</div>
<div class="grid2">
<div class="field"><label for="finalScore">내가 확정할 점수</label><select id="finalScore"><option value="">미입력</option><option value="-2">-2 강한 부정</option><option value="-1">-1 약한 부정</option><option value="0">0 중립</option><option value="1">+1 약한 긍정</option><option value="2">+2 강한 긍정</option></select></div>
<div class="field"><label>최종 사전 반영 여부</label><div class="hint" id="eligibility"></div></div>
</div>
<div class="field"><label for="reason">내 선정 근거 / 메모</label><textarea id="reason" placeholder="어떤 리뷰 문맥을 근거로 감성어라고 판단했는지 적으세요."></textarea></div>
<div class="flex"><button id="previousWord">이전 후보</button><button id="nextWord">다음 후보</button></div>
</div>
</section>
</div>
</main>
<script>
"use strict";
// Python이 파일 생성 시 후보 데이터를 이 JSON 자리로 삽입한다.
const INITIAL_ROWS = __CANDIDATE_JSON__;
const rows = INITIAL_ROWS.map(r => ({...r}));
const keyOf = r => String(r.token) + "\t" + String(r.pos);
const rowMap = new Map(rows.map(r => [keyOf(r), r]));
let selectedKey = null;
let page = 0;
const PAGE_SIZE = 45;
const $ = id => document.getElementById(id);
const isValidScore = v => ["-2","-1","1","2"].includes(String(v));
const isEligible = r => r.decision === "add" && r.role === "sentiment" && isValidScore(r.final_score) && String(r.reason || "").trim() !== "" && !r.knu_exact_word_exists;
const toNumber = (v, fallback=0) => { const n = Number(v); return Number.isFinite(n) ? n : fallback; };
function notice(t, isError=false){ const e=$("notice"); e.textContent=t; e.style.color=isError ? "#ab3f47" : "#176956"; }
function setText(id, val){ $(id).textContent = val == null || val === "" ? "미입력" : String(val); }
function filteredRows(){
  const q=$("search").value.trim().toLowerCase(), pos=$("posFilter").value, state=$("stateFilter").value;
  const out=rows.filter(r=>{
    if(q && !String(r.token).toLowerCase().includes(q)) return false;
    if(pos && r.pos!==pos) return false;
    if(state==="suggested") return r.suggested_score!=="" && r.suggested_score!=null;
    if(state!=="all" && r.decision!==state) return false;
    return true;
  });
  const sort=$("sort").value;
  out.sort((a,b)=>sort==="alpha" ? String(a.token).localeCompare(String(b.token),"ko") :
    sort==="positive" ? toNumber(b.log2_class_ratio)-toNumber(a.log2_class_ratio) :
    sort==="negative" ? toNumber(a.log2_class_ratio)-toNumber(b.log2_class_ratio) :
    toNumber(b.review_count)-toNumber(a.review_count));
  return out;
}
function makeCell(tr, value, cls="") {const td=document.createElement("td");td.textContent=String(value??""); if(cls)td.className=cls;tr.appendChild(td);}
function renderList(){
  const visible=filteredRows(), tbody=$("candidateTable");tbody.replaceChildren();
  const pages=Math.max(1,Math.ceil(visible.length/PAGE_SIZE)); page=Math.max(0,Math.min(page,pages-1));
  for(const r of visible.slice(page*PAGE_SIZE,(page+1)*PAGE_SIZE)){
    const tr=document.createElement("tr");tr.setAttribute("data-key",keyOf(r));if(keyOf(r)===selectedKey)tr.className="current";
    makeCell(tr,r.token);makeCell(tr,r.pos);makeCell(tr,toNumber(r.review_count).toLocaleString());
    makeCell(tr,r.suggested_score==="" ? "—" : r.suggested_score,"num");
    makeCell(tr,r.final_score==="" ? "—" : r.final_score,"num");
    makeCell(tr,r.decision==="add"?"추가":r.decision==="reject"?"제외":"보류");
    tr.addEventListener("click",()=>showEditor(keyOf(r)));tbody.appendChild(tr);
  }
  $("visibleCount").textContent=visible.length+"개";
  $("pageInfo").textContent=`${page+1} / ${pages} 페이지`;
  $("prevPage").disabled=page===0; $("nextPage").disabled=page+1>=pages;
  $("countAll").textContent=rows.length.toLocaleString();
  $("countSuggested").textContent=rows.filter(r=>r.suggested_score!=="" && r.suggested_score!=null).length;
  const eligible=rows.filter(isEligible).length;
  $("countSelected").textContent=eligible;
  $("countRemaining").textContent=Math.max(0,30-eligible);
}
function examples(el, list){el.replaceChildren();if(!list || !list.length){const p=document.createElement("p");p.textContent="수집된 예문 없음";el.appendChild(p);return;}
  for(const text of list){const p=document.createElement("p");p.textContent=String(text);el.appendChild(p);}}
function showEditor(key){
  selectedKey=key;const r=rowMap.get(key);if(!r)return;
  $("emptyEditor").hidden=true;$("editor").hidden=false;
  setText("wordTitle",r.token);setText("posTitle",r.pos);
  $("metrics").textContent=`전체 ${toNumber(r.token_count).toLocaleString()}회 · 출현 리뷰 ${toNumber(r.review_count).toLocaleString()}건 · 긍정 ${toNumber(r.positive_review_hits).toLocaleString()}건 / 부정 ${toNumber(r.negative_review_hits).toLocaleString()}건 · log2 비율 ${toNumber(r.log2_class_ratio).toFixed(2)}`;
  $("knuWarning").textContent=r.knu_exact_word_exists?"주의: KNU 원본에 동일 문자열이 있습니다. 확장판으로 내보낼 수 없습니다.":"";
  examples($("positiveExamples"),r.positive_examples);examples($("negativeExamples"),r.negative_examples);
  setText("suggestedScore",r.suggested_score);setText("suggestedRole",r.suggested_role);setText("suggestedReason",r.suggested_reason);
  $("decision").value=r.decision||"review";$("role").value=r.role||"";
  $("finalScore").value=r.final_score==null?"":String(r.final_score);
  $("reason").value=r.reason||"";
  $("eligibility").textContent=isEligible(r)?"반영 가능": "미확정: 감성어 + 비영점 점수 + 근거 + 추가 승인 필요";
  renderList();
}
function applyEdits(){
  if(!selectedKey)return;
  const r=rowMap.get(selectedKey);
  r.decision=$("decision").value;r.role=$("role").value;r.final_score=$("finalScore").value;r.reason=$("reason").value;
  $("eligibility").textContent=isEligible(r)?"반영 가능":"미확정: 감성어 + 비영점 점수 + 근거 + 추가 승인 필요";
  renderList();
}
for(const id of ["decision","role","finalScore"]){$(id).addEventListener("change",applyEdits);}
$("reason").addEventListener("input",applyEdits);
$("applySuggestion").addEventListener("click",()=>{
  if(!selectedKey)return; const r=rowMap.get(selectedKey);
  if(!isValidScore(r.suggested_score) || r.suggested_role!=="sentiment"){
    notice("추천 결과가 감성어(+/-1,+/-2)가 아닙니다. 직접 검토 후 판단하세요.",true);return;
  }
  $("decision").value="add";$("role").value="sentiment";$("finalScore").value=String(r.suggested_score);
  if(!$("reason").value.trim())$("reason").value="AI 초안 검토: "+(r.suggested_reason||"");
  applyEdits();notice("AI 추천을 입력란에 복사했습니다. 예문과 근거를 검토한 뒤 확정하세요.");
});
function moveSelection(d){const visible=filteredRows();const i=visible.findIndex(r=>keyOf(r)===selectedKey);if(i<0)return;const j=Math.max(0,Math.min(visible.length-1,i+d));page=Math.floor(j/PAGE_SIZE);showEditor(keyOf(visible[j]));}
$("previousWord").addEventListener("click",()=>moveSelection(-1));$("nextWord").addEventListener("click",()=>moveSelection(1));
for(const id of ["search","posFilter","stateFilter","sort"]){$(id).addEventListener(id==="search"?"input":"change",()=>{page=0;renderList();});}
$("prevPage").addEventListener("click",()=>{page--;renderList();});$("nextPage").addEventListener("click",()=>{page++;renderList();});
// CSV 표준: 따옴표 이스케이프, 줄바꿈 포함 셀, UTF-8 BOM 모두 취급한다.
function csvParse(s){
  s=s.replace(/^\uFEFF/,""); const out=[];let row=[],v="",q=false;
  for(let i=0;i<s.length;i++){const c=s[i];
    if(q){if(c==='"' && s[i+1]==='"'){v+='"';i++;}else if(c==='"'){q=false;}else{v+=c;}}
    else if(c==='"'){q=true;}
    else if(c===','){row.push(v);v="";}
    else if(c==='\n'){row.push(v);out.push(row);row=[];v="";}
    else if(c==='\r'){if(s[i+1]==='\n')continue;row.push(v);out.push(row);row=[];v="";}
    else{v+=c;}}
  if(q)throw new Error("닫히지 않은 CSV 따옴표");
  if(v!==""||row.length){row.push(v);out.push(row);}
  if(!out.length)return [];
  const header=out.shift();return out.filter(r=>r.some(x=>x!=="")).map(r=>Object.fromEntries(header.map((h,i)=>[h,r[i]??""])));
}
const csvEscape=v=>{const s=String(v??"");return /[",\r\n]/.test(s)?'"'+s.replaceAll('"','""')+'"':s;};
function csvDownload(name,arr,cols){
  const text="\uFEFF"+cols.join(",")+"\r\n"+arr.map(r=>cols.map(c=>csvEscape(r[c])).join(",")).join("\r\n")+"\r\n";
  const url=URL.createObjectURL(new Blob([text],{type:"text/csv;charset=utf-8"}));
  const a=document.createElement("a");a.href=url;a.download=name;document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),1000);
}
const OUTPUT_COLUMNS=["token","pos","token_count","review_count","positive_review_hits","negative_review_hits","log2_class_ratio","match_status","knu_exact_word_exists","suggested_score","suggested_role","suggested_reason","decision","role","final_score","reason"];
$("saveReview").addEventListener("click",()=>{csvDownload("domain_lexicon_reviewed.csv",rows,OUTPUT_COLUMNS);notice("전체 검토 CSV를 다운로드했습니다. 작업 중 백업 파일로 보관하세요.");});
$("saveSelected").addEventListener("click",()=>{
  const invalid=rows.filter(r=>r.decision==="add" && !isEligible(r));
  if(invalid.length){notice(`추가 승인된 항목 중 ${invalid.length}개가 조건(점수/근거/역할/KNU 중복)을 만족하지 않습니다.`,true);return;}
  const picked=rows.filter(isEligible);
  const ambiguous=new Set();const seen=new Set();for(const r of picked){if(seen.has(r.token))ambiguous.add(r.token);seen.add(r.token);}
  if(ambiguous.size){notice("동일 문자열에 품사별 여러 점수를 지정했습니다: "+[...ambiguous].join(", ")+" (KNU의 word 키 충돌)",true);return;}
  if(picked.length<30){notice(`현재 ${picked.length}개입니다. 감성어 30개 이상을 검토하고 승인하세요.`,true);return;}
  csvDownload("selected_domain_sentiment.csv",picked,OUTPUT_COLUMNS);notice(`${picked.length}개 선정 CSV 다운로드 완료. data/outputs/sentiment_vocab/reviewed/ 로 복사하세요.`);
});
async function readCSVFile(file){if(!file)return [];return csvParse(await file.text());}
$("aiFile").addEventListener("change",async e=>{
  try{const input=await readCSVFile(e.target.files[0]);let changed=0,skipped=0;
    for(const item of input){const r=rowMap.get(keyOf(item));if(!r){skipped++;continue;}
      const score=String(item.suggested_score??"").trim();
      if(score && !["-2","-1","0","1","2"].includes(score))throw new Error(`${r.token}: 추천 점수는 -2..2 또는 공란만 허용`);
      r.suggested_score=score;r.suggested_role=String(item.suggested_role??"").trim();r.suggested_reason=String(item.suggested_reason??"").trim();changed++;
    }
    renderList();if(selectedKey)showEditor(selectedKey);notice(`AI 추천 ${changed}행 반영, 매칭 실패 ${skipped}행. 기존 수동 검토 기록은 유지했습니다.`);
  }catch(err){notice("추천 CSV 읽기 실패: "+err.message,true);}e.target.value="";
});
$("resumeFile").addEventListener("change",async e=>{
  try{const input=await readCSVFile(e.target.files[0]);let changed=0;
    for(const item of input){const r=rowMap.get(keyOf(item));if(!r)continue;
      for(const c of ["decision","role","final_score","reason","suggested_score","suggested_role","suggested_reason"]){if(Object.hasOwn(item,c))r[c]=String(item[c]??"");}
      changed++;
    }renderList();if(selectedKey)showEditor(selectedKey);notice(`이전 검토 CSV에서 ${changed}행 복원했습니다.`);
  }catch(err){notice("검토 CSV 복원 실패: "+err.message,true);}e.target.value="";
});
const posList=[...new Set(rows.map(r=>r.pos))].sort();for(const pos of posList){const op=document.createElement("option");op.value=pos;op.textContent=pos;$("posFilter").appendChild(op);}
renderList();if(rows.length){selectedKey=keyOf(filteredRows()[0]||rows[0]);showEditor(selectedKey);}
</script>
</body></html>

'''

## 5. HTML 파일 생성 및 브라우저에서 열기

- 기본 실행은 `data/outputs/sentiment_vocab/domain_sentiment_review.html`을 생성한다.
- HTML에는 **모든 후보**와 수집된 긍정·부정 예문을 포함한다. AI에 보낼 중간 CSV에는 그중 일부만 실린다.
- `OPEN_BROWSER=True`면 현재 PC 기본 브라우저를 여는 것을 시도한다. Linux 클라우드처럼 GUI가 없으면 출력 경로를 통해 파일을 직접 열면 된다.
- HTML 파일은 입력 후보를 새로 읽어 생성하므로 재생성할 수 있다. 단, **브라우저에서 입력한 검토 내용은 다운로드한 CSV로 백업**해야 한다.

In [ ]:
OPEN_BROWSER = True
HTML_PATH = OUT / "domain_sentiment_review.html"

def clean_json_value(x):
    # pandas NaN, numpy scalar 등이 JS의 null/숫자/문자열이 되도록 정리한다.
    if pd.isna(x):
        return None
    if hasattr(x, "item"):
        x = x.item()
    return x

records = []
for item in curation.to_dict(orient="records"):
    r = {k: clean_json_value(v) for k, v in item.items()}
    token, pos = str(r["token"]), str(r["pos"])
    previous_score = r.get("custom_score")
    records.append({
        "token": token, "pos": pos,
        "token_count": r.get("token_count"), "review_count": r.get("review_count"),
        "positive_review_hits": r.get("positive_review_hits"),
        "negative_review_hits": r.get("negative_review_hits"),
        "log2_class_ratio": r.get("log2_class_ratio"),
        "match_status": r.get("match_status"),
        "knu_exact_word_exists": bool(r.get("knu_exact_word_exists")),
        "positive_examples": examples_for(token, pos, 1),
        "negative_examples": examples_for(token, pos, 0),
        # 3단계 수작업 검토 기록을 초기 상태로 가져온다.
        "decision": r.get("decision") if r.get("decision") in ("add", "reject") else "review",
        "role": r.get("role") or "",
        "final_score": "" if previous_score is None else str(int(previous_score)) if float(previous_score).is_integer() else str(previous_score),
        "reason": r.get("reason") or "",
        "suggested_score": "", "suggested_role": "", "suggested_reason": "",
    })

# </script> 같은 문자열이 리뷰에 들어가더라도 script 태그를 조기 종료하지 못하도록 이스케이프.
encoded = json.dumps(records, ensure_ascii=False, allow_nan=False)
encoded = encoded.replace("&", "\\u0026").replace("<", "\\u003c").replace(">", "\\u003e")
HTML_PATH.write_text(HTML_TEMPLATE.replace("__CANDIDATE_JSON__", encoded), encoding="utf-8")
print("브라우저 파일:", HTML_PATH)
print("수록 후보:", len(records))
if OPEN_BROWSER:
    opened = webbrowser.open(HTML_PATH.as_uri())
    print("기본 브라우저 열기 요청:", "전송됨" if opened else "미지원 (파일을 직접 열어주세요)")

## 6. 검토 CSV를 다운로드한 뒤 병합 준비

**브라우저가 작성한 파일은 자동으로 노트북에 전달되지 않는다.** 아래 파일을 수동으로 옮겨야 한다.

- 브라우저에서 `선정 30개 이상 CSV 다운로드` 클릭 → `selected_domain_sentiment.csv` 생성.
- 다운로드 폴더에서 아래 경로로 복사:
  - `4-2/data/outputs/sentiment_vocab/reviewed/selected_domain_sentiment.csv`
- 검토 전체를 보존하려면 `domain_lexicon_reviewed.csv`도 함께 백업한다.
- 저장된 CSV의 **사람이 승인한 `decision=add`, `role=sentiment`, `final_score`**를 사용한다. AI 추천 자체는 최종 승인으로 인정하지 않는다.
- 마지막 셀은 **원본 KNU를 덮어쓰지 않으며**, `data/lexicons/ko/derived/`에 버전별 새 파일을 만든다.

### 주의: STEM과 사전 매칭

- 여기서 선택한 `token`은 3단계의 **Okt `stem=True` 기준**이다.
- 새 JSON은 KNU와 호환되는 필드(`word`, `word_root`, `polarity`)로 저장하지만, **실제 감성 분석기에서 Okt 형태소를 같은 기준으로 정규화해 매칭하는 로직은 이후에 구현**해야 한다.
- 같은 문자열이 원본 KNU에 이미 있거나, 다른 품사로 중복 선택되면 덮어쓰기 대신 **오류로 중단**한다.

In [ ]:
REVIEWED_DIR = OUT / "reviewed"
SELECTION_CSV = REVIEWED_DIR / "selected_domain_sentiment.csv"

if not SELECTION_CSV.is_file():
    print("아직 선정 CSV가 없습니다.")
    print("1) HTML에서 감성어 30개 이상 승인")
    print("2) '선정 30개 이상 CSV 다운로드' 클릭")
    print("3) 다운로드한 파일을 다음 경로로 복사:", SELECTION_CSV)
else:
    preview = pd.read_csv(SELECTION_CSV, encoding="utf-8-sig", dtype="string", keep_default_na=False)
    print("선정 CSV 수록 행:", len(preview))
    display(preview[["token", "pos", "final_score", "reason"]].head(10))

## 7. KNU 확장 사전을 **새 버전으로 생성**

- 원본: `data/lexicons/ko/SentiWord_info.json` **읽기 전용**.
- 결과: `data/lexicons/ko/derived/SentiWord_info_extended_v001.json`부터 증가.
- 추가 검토 기록: 동일 버전의 `custom_entries_v001.csv`와 `metadata_v001.json`.
- **동일 경로 덮어쓰기 금지:** 이미 `v001`이 있으면 `v002`를 만든다.
- 조건: 감성어로 승인된 **서로 다른 문자열 30개 이상**, `-2/-1/1/2` 정수 점수, 근거 필수, 원본 문자열 중복 없음.
- `custom_entries`에 **품사·추천 점수·최종 점수·근거**를 남긴다. KNU 형식의 본 사전에는 품사 필드가 없기 때문이다.
- 실제 감성 분석기의 적용/성능 평가는 5단계에서 수행한다.

In [ ]:
def create_extended_knu(selection_csv=SELECTION_CSV, *, min_additions=30):
    if not selection_csv.is_file():
        print("선정 CSV가 없어 확장 사전을 생성하지 않았습니다:", selection_csv)
        return None

    selected = pd.read_csv(selection_csv, encoding="utf-8-sig", dtype="string", keep_default_na=False)
    required = {"token", "pos", "decision", "role", "final_score", "reason"}
    if not required.issubset(selected.columns):
        raise ValueError(f"선정 CSV 컬럼 누락: {sorted(required-set(selected.columns))}")

    approved = selected.loc[
        selected["decision"].eq("add") & selected["role"].eq("sentiment")
    ].copy()
    # 빈 문자열, 중복, 점수의 정수 여부를 명시적으로 검증한다.
    if len(approved) < min_additions:
        raise ValueError(f"선정 감성어가 {len(approved)}개입니다. 최소 {min_additions}개 필요")
    if approved[["token", "pos"]].duplicated().any():
        raise ValueError("(token,pos)가 중복 선택되었습니다.")
    if approved["token"].duplicated().any():
        raise ValueError("동일 문자열이 다른 품사로 중복 선택되었습니다. KNU word 키는 품사를 구분하지 않습니다.")
    if approved["token"].str.strip().eq("").any() or approved["reason"].str.strip().eq("").any():
        raise ValueError("단어 또는 선정 근거가 비어 있습니다.")
    if not approved["final_score"].isin(["-2", "-1", "1", "2"]).all():
        raise ValueError("최종 점수는 -2, -1, 1, 2 중 하나여야 합니다.")
    overlap = set(approved["token"]) & knu_words
    if overlap:
        raise ValueError(f"KNU 원본에 같은 문자열 존재 (중복 병합 금지): {sorted(overlap)[:15]}")

    # KNU의 기존 항목은 그대로 두고 3개 필드 형식으로 새 항목을 뒤에 추가한다.
    new_records = [{"word": str(r.token), "word_root": str(r.token), "polarity": str(r.final_score)}
                   for r in approved.itertuples(index=False)]
    extended = knu_records + new_records
    derived = KNU_JSON.parent / "derived"
    derived.mkdir(parents=True, exist_ok=True)
    version = 1
    while any((derived / f"{prefix}_v{version:03d}.{ext}").exists() for prefix, ext in [
        ("SentiWord_info_extended", "json"), ("custom_entries", "csv"), ("metadata", "json")
    ]):
        version += 1
    json_path = derived / f"SentiWord_info_extended_v{version:03d}.json"
    csv_path = derived / f"custom_entries_v{version:03d}.csv"
    meta_path = derived / f"metadata_v{version:03d}.json"
    metadata = {
        "original": str(KNU_JSON.relative_to(ROOT)),
        "original_sha256": hashlib.sha256(KNU_JSON.read_bytes()).hexdigest(),
        "selection_csv": str(selection_csv.relative_to(ROOT)),
        "selection_csv_sha256": hashlib.sha256(selection_csv.read_bytes()).hexdigest(),
        "original_entry_count": len(knu_records),
        "custom_entry_count": len(new_records),
        "extended_entry_count": len(extended),
        "score_policy": "-2,-1,1,2; human approved",
        "normalization": "Okt norm=True, stem=True; word_root copies stem token; downstream matching required",
    }

    # 'x' 모드는 기존 파일이 있으면 실패하므로 실수로 덮어쓰지 않는다.
    with json_path.open("x", encoding="utf-8") as fp:
        json.dump(extended, fp, ensure_ascii=False, indent=2)
    with csv_path.open("x", encoding="utf-8-sig", newline="") as fp:
        approved.to_csv(fp, index=False)
    with meta_path.open("x", encoding="utf-8") as fp:
        json.dump(metadata, fp, ensure_ascii=False, indent=2)
    print("확장 감성사전:", json_path.relative_to(ROOT))
    print("검토 기록:", csv_path.relative_to(ROOT))
    print("출처/해시 기록:", meta_path.relative_to(ROOT))
    print(f"원본 {len(knu_records):,} + 신규 {len(new_records)} = {len(extended):,}개 항목")
    return json_path

# 같은 셀을 재실행하면 새로운 번호의 버전이 생성된다. 필요할 때만 실행한다.
CREATE_EXTENDED_NOW = False  # CSV의 30개 선정을 확인한 후 True로 변경
if CREATE_EXTENDED_NOW:
    created_path = create_extended_knu()
else:
    print("안전 모드: 아직 병합하지 않았습니다. CREATE_EXTENDED_NOW=True로 변경 후 실행하세요.")

## 완료 기준과 다음 단계

- [ ] 후보 CSV / 예문 JSON을 읽고 HTML을 생성했다.
- [ ] `ai_review_request_*.csv`를 ChatGPT에 제공해 **추천 점수·역할·이유**를 받아왔다.
- [ ] 브라우저에서 추천 CSV를 불러오고, 후보별 문맥을 확인하며 30개 이상을 승인했다.
- [ ] `selected_domain_sentiment.csv`를 다운로드해 `data/outputs/sentiment_vocab/reviewed/`로 옮겼다.
- [ ] 마지막 셀에서 새 버전 KNU 확장 JSON과 출처 메타데이터를 만들었다.

### 이후 작업

- **실제 감성 분석 로직:** KNU 원본 및 확장 표현을 Okt STEM 결과에 매칭하는 방법 결정.
- **규칙:** 강조어·부정어 처리 및 이중 부정 구현.
- **평가:** NSMC에서 기본 KNU vs 확장 KNU의 Accuracy/F1 비교. 단어 선정 과정에 사용한 리뷰와 최종 평가 리뷰는 분리한다.